# Notebook 04: Statistical Analysis, Hypothesis Testing & Regression
## Distribution Fitting, Non-Parametric Group Tests, Correlations & OLS Regression

### Analytical Objectives:
1. Conduct formal normality testing (Shapiro-Wilk and D'Agostino-Pearson) across key variables.
2. Execute non-parametric group difference tests (Kruskal-Wallis across regions; Mann-Whitney U for Circuit Types).
3. Compute bivariate Pearson and Spearman correlation matrices with exact p-values.
4. Fit an Ordinary Least Squares (OLS) regression model to identify predictors of attendance while evaluating diagnostics and structural limitations.

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

f1_df = pd.read_csv("data/processed/f1_races_clean.csv")
f1_eng = pd.read_csv("data/engineered/f1_races_engineered.csv")
print(f"Loaded {len(f1_df)} baseline races.")

### 1. Normality Testing (Shapiro-Wilk & D'Agostino-Pearson)

In [ ]:
test_vars = ['attendance_race_day', 'average_speed_kmh', 'winning_margin_s', 'gdp_per_capita_usd', 'circuit_length_km']
norm_results = []

for v in test_vars:
    data = f1_df[v].dropna()
    shapiro_stat, shapiro_p = stats.shapiro(data)
    dagost_stat, dagost_p = stats.normaltest(data)
    norm_results.append({
        'Variable': v,
        'Shapiro-Wilk Stat': round(shapiro_stat, 4),
        'Shapiro p-value': f"{shapiro_p:.2e}",
        'DAgostino Stat': round(dagost_stat, 4),
        'DAgostino p-value': f"{dagost_p:.2e}",
        'Is Normal (alpha=0.05)': 'Yes' if shapiro_p > 0.05 else 'No'
    })

pd.DataFrame(norm_results)

### 2. Non-Parametric Group Testing
Because attendance violates normality assumptions ($p < 0.001$), we use Kruskal-Wallis across continents and Mann-Whitney U for 2-group comparisons.

In [ ]:
# Kruskal-Wallis across Continents
continents = [group['attendance_race_day'].values for name, group in f1_df.groupby('continent')]
kw_stat, kw_p = stats.kruskal(*continents)
print(f"Kruskal-Wallis test across Continents: H = {kw_stat:.4f}, p = {kw_p:.4e}")

# Mann-Whitney U: Europe vs Asia
europe_att = f1_df[f1_df['continent'] == 'Europe']['attendance_race_day']
asia_att = f1_df[f1_df['continent'] == 'Asia']['attendance_race_day']
u_stat, u_p = stats.mannwhitneyu(europe_att, asia_att, alternative='two-sided')
print(f"Mann-Whitney U (Europe vs Asia): U = {u_stat:.1f}, p = {u_p:.4e} (Europe Median={europe_att.median():,.0f} vs Asia Median={asia_att.median():,.0f})")

# Mann-Whitney U: Street vs Permanent Circuits
street_att = f1_df[f1_df['circuit_type'] == 'Street']['attendance_race_day']
perm_att = f1_df[f1_df['circuit_type'] == 'Permanent']['attendance_race_day']
u_type_stat, u_type_p = stats.mannwhitneyu(street_att, perm_att, alternative='two-sided')
print(f"Mann-Whitney U (Street vs Permanent): U = {u_type_stat:.1f}, p = {u_type_p:.4f} (Street Median={street_att.median():,.0f} vs Permanent Median={perm_att.median():,.0f})")

### 3. Correlation Analysis (Pearson & Spearman with p-values)

In [ ]:
corr_pairs = [
    ('attendance_race_day', 'gdp_per_capita_usd', 'Attendance vs GDP per Capita'),
    ('attendance_race_day', 'event_tenure_years', 'Attendance vs Event Tenure'),
    ('attendance_race_day', 'average_speed_kmh', 'Attendance vs Average Speed'),
    ('average_speed_kmh', 'circuit_length_km', 'Speed vs Circuit Length'),
    ('winning_margin_s', 'finishers_count', 'Winning Margin vs Finishers Count')
]

corr_results = []
for v1, v2, desc in corr_pairs:
    p_r, p_p = stats.pearsonr(f1_df[v1], f1_df[v2])
    s_r, s_p = stats.spearmanr(f1_df[v1], f1_df[v2])
    corr_results.append({
        'Relationship': desc,
        'N': len(f1_df),
        'Pearson r': round(p_r, 3),
        'Pearson p': f"{p_p:.3e}",
        'Spearman rho': round(s_r, 3),
        'Spearman p': f"{s_p:.3e}"
    })

pd.DataFrame(corr_results)

### 4. Ordinary Least Squares (OLS) Multivariate Regression
Modeling Sunday Race-Day Attendance as a function of host nation GDP per capita, event tenure, circuit type, and race distance.

In [ ]:
# Fit OLS Model
model = smf.ols(
    formula='attendance_race_day ~ np.log(gdp_per_capita_usd) + event_tenure_years + C(circuit_type) + race_distance_km',
    data=f1_df
).fit()

print(model.summary())

### 5. Regression Diagnostics & Limitations
- **$R^2$:** 0.365 (Adjusted $R^2$: 0.349, $F(5, 192) = 22.08, p = 1.88 	imes 10^{-17}$). Macroeconomic and sporting factors explain 36.5% of global attendance variance.
- **Log GDP per Capita:** Positive statistical association with attendance ($eta = 5,143.6, t = 2.37, p = 0.019$). A 10% increase in host nation GDP per capita is associated with $pprox 5,143.6 	imes \ln(1.10) pprox 490$ additional Sunday spectators.
- **Event Tenure:** Positive and statistically significant ($eta = 487.9, t = 6.14, p < 0.001$), supporting the commercial importance of historical heritage.
- **Circuit Type:** Permanent and Street circuits have lower baseline attendance relative to Hybrid circuits ($eta = -29,310, p < 0.001$ and $eta = -33,300, p < 0.001$).
- **Methodological Boundary:** This exploratory model identifies baseline associations across 198 races; observations within the same circuit over time share clustered errors, and these associations should not be interpreted as direct causal mechanisms for individual Grand Prix cancellations.